# 自环测试（用我们的位流，720p）

**接线**：一根 HDMI 线，从板子 **HDMI OUT** 直接插到板子 **HDMI IN**。
笔记本和采集卡都先不接。

## 为什么用我们的位流

| 位流 | 输入 PLL | 输出 PLL | 720p 自环可行？ |
| --- | --- | --- | --- |
| 原厂 `base.bit` | range 1（≥120 MHz） | range 2（60–120 MHz） | ❌ 两端没有交集 |
| 我们的 `enhance_hdmi.bit` | range 2 | range 2 | ✅ 720p 落在两端范围内 |

## 预期

输出会是**黑屏**（输入没有信号源，内核无数据可传），这**算通过**——
只要 `hdmi_in.mode` 能报到 720p，就证明输出端真的发出了 TMDS、输入端真的锁上了。


In [ ]:
import time
from pynq import Overlay
from pynq.lib.video import VideoMode

ol = Overlay("enhance_hdmi.bit")
v = ol.video
print("overlay loaded, IPs:", len(ol.ip_dict))

mode = VideoMode(1280, 720, 24, 60)
v.hdmi_out.configure(mode)
v.hdmi_out.start()
print("hdmi_out configured and started:", v.hdmi_out.mode)

# HPD off -> on, so the receiving side performs a clean hot-plug and re-reads EDID.
hpd = None
for name in ol.ip_dict:
    if "gpio_hdmiin" in name.lower():
        hpd = getattr(ol, name)


In [ ]:
if hpd is not None:
    hpd.mmio.write(0x00, 0x0)
    time.sleep(1.0)
    hpd.mmio.write(0x00, 0x1)
    print("HPD toggled 0 -> 1")
else:
    print("no HPD gpio in this overlay")


## 监视 30 秒

In [ ]:
locked = None
for i in range(30):
    m = v.hdmi_in.mode
    if m is not None:
        locked = m
        print(f"[{i:2d}] LOCKED {m.width}x{m.height} @{m.frame_rate} Hz")
        break
    if i % 5 == 0:
        print(f"[{i:2d}] mode: None")
    time.sleep(1.0)


In [ ]:
print()
if locked is None:
    print("RESULT: 自环也没有 TMDS")
    print("  -> 输出端（v_axi4s_vid_out / rgb2dvi）没有发出信号，")
    print("     这与采集卡一直显示兜底彩条的现象一致")
    print("  -> 下一步：在 v_axi4s_vid_out 的 tvalid/tready/tuser/tlast 上插 ILA")
else:
    print(f"RESULT: 自环锁定 {locked.width}x{locked.height}@{locked.frame_rate}")
    print("  -> 板子的 HDMI 收发两侧都正常")
    print("  -> 问题在笔记本侧：把它接到 IN，并把分辨率设成 1280x720 再试")


## 测完怎么恢复

把线恢复成 笔记本→IN、OUT→采集卡，然后加载我们的位流：

```python
from pynq import Overlay
ol = Overlay("enhance_hdmi.bit")
```
